# RSNA Knee Abnormality Detection - CPU baseline & sample submission

- **Task:** predict the per-study probability of **12 knee MRI findings** (ACL, MCL, meniscus, OA compartments, effusion, etc.).
- **Metric:** macro-averaged ROC AUC over the twelve labels.
- **Submission:** one row per test study, confidence score per label -> file named `submission.csv`.
- **This notebook:** runs on **CPU only** and keeps the data download **minimal** (only five tiny CSV files, *not* the ~570 GB of DICOMs).

### How to run
- **Kaggle Code (recommended):** add the competition dataset, pick CPU (or GPU) accelerator, run. Data is already mounted at `/kaggle/input` -> nothing is downloaded.
- **Locally:** install `kaggle` CLI with credentials (`kaggle competitions download`), then just run the cells. Only the 5 small CSVs are fetched.

The notebook builds two CPU baselines and writes a valid `submission.csv`:
1. **Baseline A** - predict each label at its training-set prevalence (guaranteed-valid scaffold, macro-AUC ~0.5).
2. **Baseline B** - a tiny logistic-regression model on study-level MRI *series descriptors* (planes, fluid sensitivity, fat suppression). Cross-validated on the train set, this is a real (if weak) learned baseline that beats 0.5.


## 1. Data description (summary)

Each **study** = one knee MRI exam = several **series** (DICOM sequences). ~5,000 training studies, ~1,300 test studies.

| File | Contents |
|---|---|
| `train.csv` | one row per study: `StudyInstanceUID`, `PatientSex`, free-text `Report` (multilingual), 12 binary labels |
| `train_series.csv` | one row per series: study/series UIDs, `Fluid_Sensitive`, `Fat_Suppression`, `Anatomical_Plane` |
| `train_series/` | DICOMs `train_series/<Study>/<Series>/<Slice>.dcm` (20-45 slices per series) |
| `test.csv` / `test_series.csv` / `test_series/` | same, for ~1,300 test studies |
| `sample_submission.csv` | all labels set to `0.5` |

The 12 labels: `ACL`, `MCL`, `Medial Meniscus`, `Lateral Meniscus`, `Medial OA`, `Lateral OA`, `PF OA`, `Effusion`, `Synovitis`, `Baker's`, `Contusion`, `Fracture`.


In [ ]:
import os
import sys
import subprocess

import numpy as np
import pandas as pd

COMPETITION = "rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
          "Medial OA", "Lateral OA", "PF OA", "Effusion",
          "Synovitis", "Baker's", "Contusion", "Fracture"]

ON_KAGGLE = os.path.exists("/kaggle/input")
DATA_DIR = "/kaggle/input/rsna-knee-abnormality-detection" if ON_KAGGLE else os.path.join(os.getcwd(), "data")
os.makedirs(DATA_DIR, exist_ok=True)

BASELINE_B_OK = False  # set to True if Baseline B trains successfully

print("Running on Kaggle:", ON_KAGGLE)
print("Data dir:", DATA_DIR)
print("Selected accelerator:", os.environ.get("KAGGLE_GPU_TYPE", "n/a (CPU)"))


## 2. Download the minimal data (CSVs only)

We deliberately **skip the ~570 GB of DICOM files**. This baseline needs only the five CSVs
(a few MB in total). On Kaggle they are already mounted, so nothing is downloaded there.


In [ ]:
MINIMAL_FILES = ["train.csv", "train_series.csv", "test.csv", "test_series.csv", "sample_submission.csv"]

def ensure_minimal_data():
    if ON_KAGGLE:
        print("Data is mounted on Kaggle; skipping download.")
        return
    missing = [f for f in MINIMAL_FILES if not os.path.exists(os.path.join(DATA_DIR, f))]
    if not missing:
        print("All minimal files already present:", DATA_DIR)
        return
    print("Downloading (small) missing files:", missing)
    for f in missing:
        try:
            res = subprocess.run(
                ["kaggle", "competitions", "download", "-c", COMPETITION, "-f", f, "-p", DATA_DIR],
                check=True, capture_output=True, text=True,
            )
            print("  downloaded:", f)
        except Exception as e:
            print(f"  FAILED: {f} -> {e}")
            print("  Place the files manually into", DATA_DIR, "and re-run this cell.")
            raise SystemExit("Missing competition CSVs. See message above.")

ensure_minimal_data()


In [ ]:
def load(path):
    return pd.read_csv(path, dtype={"StudyInstanceUID": str})

train = load(os.path.join(DATA_DIR, "train.csv"))
train_series = load(os.path.join(DATA_DIR, "train_series.csv"))
test = load(os.path.join(DATA_DIR, "test.csv"))
test_series = load(os.path.join(DATA_DIR, "test_series.csv"))
sample_sub = load(os.path.join(DATA_DIR, "sample_submission.csv"))


## 3. Quick look at the data

Only a small subset of training studies carries the 12 per-condition labels; the rest have a radiology `Report` you can mine for weak labels later.


In [ ]:
print("train:", train.shape, "| test:", test.shape)
print("train_series:", train_series.shape, "| test series/study approx:", train_series["StudyInstanceUID"].nunique())
print()
print("sample_submission columns:", list(sample_sub.columns))
print()
print("Are all 12 labels present in train.csv?", all(c in train.columns for c in LABELS))
print()
print("Labeled subset prevalence (positive rate):")
prev = train[LABELS].mean()
prev.to_frame("prevalence").round(3)


## 4. Baseline A - predict the training prevalence (sample submission)

For each label, assign its observed positive rate in the labeled training subset to every test study.
This produces a **valid submission** but, being constant per label, it cannot rank studies (macro-AUC ~ 0.5).
Use it as the sanity-check scaffold that the pipeline, formatting, and `submission.csv` are correct.


In [ ]:
prevalence = train[LABELS].mean()

submission_a = pd.DataFrame({"StudyInstanceUID": test["StudyInstanceUID"]})
for lab in LABELS:
    submission_a[lab] = prevalence[lab]

print("Baseline A built for", len(submission_a), "test studies.")
submission_a.head()


## 5. Baseline B - tiny CPU model on MRI series descriptors

Even though we skip the images, `train_series.csv` tells us *which* MRI sequences each study contains
(fluid-sensitive? fat-suppressed? which anatomical plane?). These differ across studies, so a small
**logistic regression** on those counts can rank studies and typically beats 0.5 macro-AUC.

This runs in seconds on CPU and needs no extra data. Unlabeled training studies (NaN in the 12
labels) are dropped per label, so only the labeled subset is used to fit.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def series_features(series_df):
    uid = series_df["StudyInstanceUID"]
    feats = pd.DataFrame({
        "n_series": series_df.groupby(uid).size(),
        "n_fluid": series_df.groupby(uid)["Fluid_Sensitive"].sum(),
        "n_fatsat": series_df.groupby(uid)["Fat_Suppression"].sum(),
    })
    feats["n_nonfluid"] = feats["n_series"] - feats["n_fluid"]
    planes = pd.get_dummies(series_df["Anatomical_Plane"], prefix="plane")
    planes[uid.name] = series_df["StudyInstanceUID"].values
    plane_counts = planes.groupby(uid.name).sum()
    feats = feats.join(plane_counts, how="outer").fillna(0)
    return feats

X_train = series_features(train_series)
X_test = series_features(test_series)

# Align to the study tables (a few studies may lack series rows -> zeros).
X_train = X_train.reindex(train["StudyInstanceUID"]).fillna(0)
X_test = X_test.reindex(test["StudyInstanceUID"]).fillna(0)
y_train = train.set_index("StudyInstanceUID")[LABELS].reindex(X_train.index)

def cv_auc(clf, X, y, n_splits=5, seed=42):
    min_class = int(y.value_counts().min())
    n_splits = min(n_splits, min_class) if min_class >= 2 else 0
    if n_splits < 2:
        return np.nan
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    for tr, va in skf.split(X, y):
        if y.iloc[tr].nunique() < 2:
            oof[va] = prevalence[y.name]
            continue
        clf.fit(X.iloc[tr], y.iloc[tr])
        oof[va] = clf.predict_proba(X.iloc[va])[:, 1]
    if y.nunique() < 2:
        return np.nan
    return roc_auc_score(y, oof)

cv_scores = {}
pred_test = pd.DataFrame(index=X_test.index)
for lab in LABELS:
    clf = LogisticRegression(max_iter=1000)
    y = y_train[lab]
    labeled = y.notna()
    if labeled.sum() < 2 or y[labeled].nunique() < 2:
        print(f"  {lab}: too few labeled positives -> falling back to prevalence")
        pred_test[lab] = prevalence[lab]
        cv_scores[lab] = np.nan
        continue
    cv_scores[lab] = cv_auc(clf, X_train[labeled], y[labeled])
    clf.fit(X_train[labeled], y[labeled])
    pred_test[lab] = clf.predict_proba(X_test)[:, 1]

cv_series = pd.Series(cv_scores, name="CV AUC")
print(cv_series.round(3).to_frame())
print("macro CV AUC:", round(float(np.nanmean(cv_series.values)), 4))
BASELINE_B_OK = True


In [ ]:
# Pick the best available baseline, then write the official submission.
if BASELINE_B_OK:
    pred = pred_test.copy()
    pred.insert(0, "StudyInstanceUID", pred.index)
    pred = pred.reset_index(drop=True)
    print("Using Baseline B (series-descriptor logistic regression).")
else:
    pred = submission_a.copy()
    print("Using Baseline A (prevalence constant).")

# Enforce the exact column set and order of sample_submission.csv.
submission = pred[sample_sub.columns].copy()

assert list(submission.columns) == list(sample_sub.columns), "column mismatch"
assert len(submission) == len(test), "row-count mismatch"
assert submission["StudyInstanceUID"].is_unique, "duplicate StudyInstanceUID"

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv with", len(submission), "rows x", submission.shape[1], "cols.")
submission.head()


In [ ]:
# Quick check: submission.csv round-trips and is well-formed.
chk = pd.read_csv("submission.csv", dtype={"StudyInstanceUID": str})
print("round-trip ok:", chk.shape, "| labels in range [0,1]:", bool(chk[LABELS].min().min() >= 0 and chk[LABELS].max().max() <= 1))


## 6. Bonus - peek at a real DICOM slice (on Kaggle only, zero extra download)

On Kaggle the images are already mounted, so we can read one slice with `pydicom` for free.
Locally we skip this so the download stays minimal.


In [ ]:
if ON_KAGGLE:
    import glob as _glob
    import pydicom
    import matplotlib.pyplot as plt

    study_dirs = sorted(_glob.glob(os.path.join(DATA_DIR, "train_series", "*")))
    if study_dirs:
        series_dirs = sorted(_glob.glob(os.path.join(study_dirs[0], "*")))
        slices = sorted(_glob.glob(os.path.join(series_dirs[0], "*.dcm"))) if series_dirs else []
        if slices:
            mid = slices[len(slices) // 2]
            ds = pydicom.dcmread(mid)
            px = ds.pixel_array
            print("slice:", os.path.basename(mid))
            print("shape:", px.shape, "| BitsAllocated:", ds.BitsAllocated, "| Plane:", ds.get("ImagePositionPatient"))
            plt.figure(figsize=(3, 3))
            plt.imshow(px, cmap="gray")
            plt.axis("off")
            plt.show()
        else:
            print("No slices found.")
    else:
        print("No training studies mounted.")
else:
    print("Skipping DICOM preview (not downloaded locally to keep the download small).")


## Next steps (where this baseline goes from here)

1. **Reports as weak labels** - only a subset of train studies is labeled; parse the multilingual radiology `Report` text (keyword/LLM) to pseudo-label the rest, then train on all studies.
2. **Images** - a 3D/2.5D CNN on selected series (sagittal PD, sagittal fat-sat T2, coronal T1) is the standard strong approach for this kind of competition. `pydicom` -> numpy with resampling, cropping to the knee.
3. **Series selection** - use `train_series.csv` (plane + fluid sensitivity) to pick informative sequences per finding; the logistic-regression features above hint at how much signal lives in sequence composition alone.
4. **Memory** - 570 GB of DICOMs cannot all be held in RAM; stream slices on the fly / preprocess to compact numpy volumes.
5. **Efficiency track** - this competition also rewards fast models (score/time); CPU-friendly options keep you competitive there.
